# Fase 6b — Verifikasi Dimensi Citra (menutup celah di §6.2–6.3 naskah)

**Masalah.** Diagnostik Fase 6 menghitung rasio aspek (AR) sbg `w_rel / h_rel` dari label YOLO.
Tapi `w_rel` dinormalisasi thd **lebar** citra dan `h_rel` thd **tinggi**. AR piksel sebenarnya:

    AR_px = (w_rel × W) / (h_rel × H)

Keduanya sama **hanya jika citra persegi**. Radiografi sambungan las sering berbentuk pita memanjang
→ angka elongasi (22,1% AR>5, 0,0% di DSF, korelasi r=−0,93, 85–98%, dst.) **belum terverifikasi**.

**Notebook ini juga menjawab:** apakah salinan SWRD kita sudah dipotong sliding-window (citra kecil
seragam) atau masih citra utuh — klaim di §6.2 bergantung pada ini.

**Tidak butuh GPU.** Hanya membaca header citra (PIL tidak memuat piksel), cepat.

In [1]:
import subprocess, sys, os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
for slug, dest in [("benyxu/swrd-yolo8format", "/content/swrd"),
                   ("benyxu/dsf-yolo-steel-tube-dataset", "/content/dsf")]:
    os.makedirs(dest, exist_ok=True)
    if not glob.glob(f"{dest}/**/*.txt", recursive=True):
        subprocess.run(["kaggle","datasets","download","-d",slug,"-p",dest,"--unzip"], check=True)
    print("OK", slug)

Mounted at /content/drive
OK benyxu/swrd-yolo8format
OK benyxu/dsf-yolo-steel-tube-dataset


In [2]:
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
IMGSZ = 640
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def find_image(txt):
    """labels/xxx.txt -> images/xxx.<ext>. Kembalikan None kalau tidak ketemu."""
    s = str(txt)
    for a, b in [("/labels/", "/images/"), ("labels", "images")]:
        if a in s:
            base = s.replace(a, b, 1)[:-4]
            for e in EXTS:
                for cand in (base + e, base + e.upper()):
                    if os.path.exists(cand): return cand
    return None

def collect(root, name):
    rows, miss = [], 0
    for txt in Path(root).rglob("*.txt"):
        if txt.stem in {"classes", "notes", "README"}: continue
        img = find_image(txt)
        if img is None: miss += 1; continue
        with Image.open(img) as im: W, H = im.size          # header saja, cepat
        for line in open(txt):
            p = line.split()
            if len(p) < 5: continue
            try: c, wr, hr = int(p[0]), float(p[3]), float(p[4])
            except ValueError: continue
            wpx, hpx = wr * W, hr * H                          # ukuran piksel ASLI
            sc = IMGSZ / max(W, H)                             # letterbox (skala seragam)
            rows.append(dict(dataset=name, cls=c, W=W, H=H, w_rel=wr, h_rel=hr,
                ar_lama=max(wr,hr)/max(min(wr,hr),1e-9),       # rumus Fase 6 (asumsi persegi)
                ar_px=max(wpx,hpx)/max(min(wpx,hpx),1e-9),     # rumus BENAR
                side640=np.sqrt(wpx*sc * hpx*sc)))             # sisi ekuivalen @640, letterbox benar
    print(f"{name}: {len(rows):,} objek | label tanpa citra padanan: {miss}")
    return pd.DataFrame(rows)

df = pd.concat([collect("/content/swrd", "SWRD"), collect("/content/dsf", "DSF-YOLO")], ignore_index=True)

SWRD: 18,240 objek | label tanpa citra padanan: 2
DSF-YOLO: 6,783 objek | label tanpa citra padanan: 6


## 1. Bentuk citra — persegi? sudah dipotong?

In [3]:
for ds in ["SWRD", "DSF-YOLO"]:
    im = df[df.dataset == ds].drop_duplicates(["W", "H"])[["W", "H"]]
    sub = df[df.dataset == ds]
    asp = (sub[["W","H"]].max(axis=1) / sub[["W","H"]].min(axis=1))
    print(f"\n{ds}: {len(im)} ukuran citra unik")
    print(im.value_counts().head(8).to_string())
    print(f"  rasio aspek CITRA: median={asp.median():.2f}  maks={asp.max():.2f}")
    print(f"  -> {'PERSEGI (rumus lama sah)' if asp.max()<1.01 else 'TIDAK PERSEGI -> rumus lama TIDAK sah'}")


SWRD: 1 ukuran citra unik
W    H  
640  640    1
  rasio aspek CITRA: median=1.00  maks=1.00
  -> PERSEGI (rumus lama sah)

DSF-YOLO: 12 ukuran citra unik
W     H  
958   653    1
      665    1
1178  817    1
1180  813    1
      816    1
1181  815    1
      816    1
      818    1
  rasio aspek CITRA: median=1.45  maks=1.47
  -> TIDAK PERSEGI -> rumus lama TIDAK sah


## 2. Hitung ulang angka §6.2–6.3 dengan rumus yang benar

In [4]:
from scipy.stats import pearsonr, spearmanr
out = []
for ds in ["SWRD", "DSF-YOLO"]:
    s = df[df.dataset == ds]
    for col, lbl in [("ar_lama", "LAMA (asumsi persegi)"), ("ar_px", "BENAR (piksel)")]:
        a = s[col]
        r = dict(dataset=ds, rumus=lbl, n=len(s), ar_median=a.median(), ar_p90=a.quantile(.9),
                 pct_ar_gt5=(a > 5).mean()*100, n_ar_gt5=int((a > 5).sum()))
        out.append(r)
    out[-1].update(side640_median=s.side640.median(), pct_small=(s.side640 < 32).mean()*100)
summary = pd.DataFrame(out)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

NAMES = {0:"Crack",1:"Lack_Of_Fusion",2:"Lack_Of_Penetration",3:"Porosity",4:"Slag_Inclusion"}
MAP = {"Crack":0.274,"Lack_Of_Fusion":0.145,"Lack_Of_Penetration":0.152,"Porosity":0.408,"Slag_Inclusion":0.452}
sw = df[df.dataset == "SWRD"]
pc = []
for cid, cn in NAMES.items():
    k = sw[sw.cls == cid]
    pc.append(dict(kelas=cn, n=len(k), pct_gt5_lama=(k.ar_lama>5).mean()*100,
                   pct_gt5_benar=(k.ar_px>5).mean()*100, ar_median_benar=k.ar_px.median(), mAP50_95=MAP[cn]))
pc = pd.DataFrame(pc)
print("\n" + pc.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
for col in ["pct_gt5_lama", "pct_gt5_benar"]:
    print(f"  {col:14s} Pearson r={pearsonr(pc[col], pc.mAP50_95)[0]:+.3f} (p={pearsonr(pc[col], pc.mAP50_95)[1]:.3f})"
          f"  Spearman rho={spearmanr(pc[col], pc.mAP50_95)[0]:+.3f} (p={spearmanr(pc[col], pc.mAP50_95)[1]:.3f})")

 dataset                 rumus     n  ar_median  ar_p90  pct_ar_gt5  n_ar_gt5  side640_median  pct_small
    SWRD LAMA (asumsi persegi) 18240       1.28   11.43       22.14      4038             NaN        NaN
    SWRD        BENAR (piksel) 18240       1.28   11.43       22.14      4038           39.42      40.11
DSF-YOLO LAMA (asumsi persegi)  6783       1.25    1.45        0.00         0             NaN        NaN
DSF-YOLO        BENAR (piksel)  6783       1.18    1.47        0.25        17            9.98      84.00

              kelas    n  pct_gt5_lama  pct_gt5_benar  ar_median_benar  mAP50_95
              Crack 3208         14.15          14.15             2.81      0.27
     Lack_Of_Fusion 2105         84.94          84.94             8.69      0.14
Lack_Of_Penetration 1825         97.48          97.48            13.02      0.15
           Porosity 8375          0.00           0.00             1.13      0.41
     Slag_Inclusion 2727          0.62           0.62             1.1

In [5]:
OUT = f"{DRIVE_ROOT}/analysis"; os.makedirs(OUT, exist_ok=True)
summary.to_csv(f"{OUT}/verifikasi_dimensi_ringkasan.csv", index=False)
pc.to_csv(f"{OUT}/verifikasi_dimensi_per_kelas.csv", index=False)
df.drop_duplicates(["dataset","W","H"])[["dataset","W","H"]].to_csv(f"{OUT}/verifikasi_dimensi_ukuran_citra.csv", index=False)
for f in ["ringkasan", "per_kelas", "ukuran_citra"]:
    shutil.copy(f"{OUT}/verifikasi_dimensi_{f}.csv", f"/content/verifikasi_dimensi_{f}.csv")
print("Tersimpan ke Drive analysis/ + /content/: verifikasi_dimensi_{ringkasan,per_kelas,ukuran_citra}.csv")
print("Kirim ketiga CSV ke sesi analisis.")

Tersimpan ke Drive analysis/ + /content/: verifikasi_dimensi_{ringkasan,per_kelas,ukuran_citra}.csv
Kirim ketiga CSV ke sesi analisis.


## 3. Matikan Runtime (Jalankan Terakhir)

In [6]:
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")

Semua proses selesai. Runtime akan di-disconnect otomatis...
